# The Polar Data Cube

As part of the initiative, and to facilitate easier analysis of and access to scientific Antarctica data, some of the presented datasets are combined into zarr stores, reprojected to a common 100x100m grid in the EPSG:3031 Polar projection.

The main idea is to distribute analysis ready data (ARD) for different purposes, such as machine learning, visualisation and various data analysis.

You can see how we combined the presented from the .combine notebook. Primarily we used `nearest-neighbour` interpolation. The goal was to provide single access to all the data in one place.

## Why a shared cube helps

Putting datasets on a common projection, resolution, and coordinate system reduces the amount of preprocessing needed before analysis. Instead of repeatedly handling reprojection, spatial joins, resampling, and file-format differences, you can focus on comparing variables across the same `x` and `y` grid.

This is especially useful for exploratory polar workflows where several layers need to be compared together, for example ice velocity, surface elevation change, grounding lines, calving fronts, bedrock, ice temperature, and basal melt. The cube is not a replacement for the source products; it is a practical analysis layer that makes cross-dataset questions faster to prototype.

## Content

The notebooks in this folder introduce the Antarctica data cube workflow from two angles: using prebuilt remote cubes, and understanding how multiple datasets can be aligned into a shared analysis grid to produce ARD.

1. `1_remote_cube_access.ipynb` opens the remote Zarr stores lazily with xarray, selects a small region of interest, and plots example layers without downloading the full cube.
2. `2_build_single_cube_demo.ipynb` demonstrates the construction pattern on a compact ROI by combining gridded, vector, and raster inputs onto one EPSG:3031 grid.
3. `3_large_area_processing.ipynb` shows how to work with larger cube regions using Dask.

Additionally there are notebooks that show how one can access external datasets and combine it with the cube data under `./accessing_external_datasets`

## Visualization

You can see a visualisation of the data in eodash:

https://esa-earthcode.github.io/polar-science-cluster-dashboard/



# How We Created the Cube

The cube is created by transforming multiple datasets onto one shared grid for analysis. Each dataset may start with a different format, projection, resolution, grid type, or set of dimensions. The aim is to make the datasets easy to compare without changing the meaning of the original measurements.

Each layer is opened in its original form, reprojected where needed, placed onto the common grid, resampled or interpolated to the target resolution, stored with consistent coordinates and metadata, and merged into the final cube.

## Common Target Grid

All spatial layers are aligned to one projected `x`/`y` grid. The working grid uses 100 m cells, with `x` from `-2867900` to `2867900` and `y` from `-2457900` to `2457900`, giving 57,358 columns by 49,158 rows.

The same `x, y` position refers to the same place across every variable. Coordinates are stored as regular horizontal axes in projected metres, so variables can be subset, visualised, compared, and analysed in the same spatial frame.

Some datasets already closely match the target grid. Others need to be reprojected, shifted, resampled and interpolated.

## Input Data Structures

The input data can arrive in several broad spatial forms:

- **Regular raster grids**, where values are already arranged in rows and columns.
- **Projected grids**, where data is gridded but uses a different spacing, extent, or coordinate system.
- **Geographic grids**, where data is stored against longitude and latitude and must be projected before alignment.
- **Curvilinear or non-regular grids**, where cell locations are described by two-dimensional coordinate arrays rather than simple one-dimensional axes.
- **Vector geometries**, where shapes are stored as points, lines, or polygons rather than raster cells.
- **Multi-dimensional variables**, where the same spatial grid is repeated across dimensions such as time, depth, height, band, or category.

The core step is always the same: each dataset is mapped from its original structure into the cube's shared grid structure.

## Reprojection, Resampling, and Interpolation

Before layers can be combined, they must use the same coordinate reference system, or CRS. Reprojection converts each dataset from its original CRS into the cube CRS and records that target CRS in the spatial metadata.

If a dataset has a different native resolution, its values are resampled onto the target grid. If the dataset is coarser than the cube grid, this is an **upsampling** step: multiple target cells may receive values from the same source cell or nearby source sample. Upsampling aligns the data, but it does not create new observations or increase the true information content of the original dataset.

The default interpolation method is **nearest neighbour**. Each target cell receives the value from the nearest source cell, sample point, or coordinate. This works well for masks, classes, flags, identifiers, binary layers, categorical values, and rasterised geometry outputs because it preserves the original values exactly.

For continuous variables, nearest neighbour is a simple and reproducible first choice, but it can produce blocky transitions when the original grid is much coarser than the cube grid. Other interpolation methods may be better for selected continuous variables depending on the intended analysis.

## Vector and Non-Regular Data

Vector datasets are converted into raster variables by burning their geometries onto the target grid. The burned value can be a constant, a source attribute, a class code, an identifier, or a time or measurement attribute. After rasterisation, the result behaves like any other cube variable, with the same `x` and `y` coordinates, grid spacing, and extent as the rest of the cube.

For curvilinear or non-regular grids, coordinate arrays are used to locate each value in space. Values are then assigned to the regular cube grid using the selected interpolation method, usually nearest neighbour unless a different method is chosen for that variable.

If a dataset contains extra dimensions, such as `time`, `depth`, `height`, `band`, or `category`, the spatial transformation is applied across the horizontal grid while preserving those dimensions. A static layer, a time-varying layer, and a profile layer can coexist in the same cube as long as their spatial coordinates are consistent.

## Metadata and Assembly

Each variable includes metadata that explains both the original dataset and the processing applied during cube creation. This is stored as variable-level attributes, with shared spatial metadata stored once for the common grid.

Useful metadata includes `source`, `reference`, `original_cellsize_metres`, `units`, `long_name`, `standard_name` where available, `description`, CRS/grid-mapping fields, interpolation or rasterisation method, `processing_steps`, and known assumptions or limitations.

After each dataset has been transformed onto the target grid, the processed variables are merged into a single multi-variable cube. The final cube stores shared coordinates once, then attaches each variable to those coordinates.

The cube is therefore an integration layer: it brings different datasets into one common analytical framework while keeping the processing transparent and reproducible.



## Current Assumptions and Future Refinements

We note that the current approach and any other approach to upsampling or downsampling the original data will invite distortions. In some cases the data then becomes unusable for some scientific analysis. This is why we also transform and keep the original data in a cloud-friendly format (see [1_Datasets](../1_Datasets/datasets_sumary.ipynb)) to enable other scientific workflows that do not require a whole integrated data layer. 

The current approach prioritises consistency, traceability, and reproducibility. Nearest-neighbour interpolation is used as a robust default because it preserves original values and works well for categorical or mask-like variables.

Future refinements can choose the resampling method variable by variable: nearest neighbour for masks, flags, classes, IDs, and categories; bilinear or higher-order interpolation for selected continuous fields; conservative or area-weighted resampling where totals or densities must be preserved; and separate temporal handling where variables have incompatible time structures.

